# Using Tools With Claude

This notebook demonstrates how tools let an AI assistant gather external information before answering. We will build a small mock weather tool and walk through the complete Anthropic tool-calling message flow.

## What You Will Learn

- How to call Claude with plain text and structured messages
- How to define a tool with typed parameters
- How the model requests a tool call
- How to return tool results to the model
- How the final assistant response is produced

## Setup

Load environment variables, clear cached course helper modules, then import the local wrapper classes. The Anthropic SDK will use `ANTHROPIC_API_KEY` from your environment or `.env` file.

In [1]:
import json
import os
import sys

from dotenv import load_dotenv

load_dotenv(".env")
# Refresh local course helpers after editing files in lib/.
for module_name in [name for name in list(sys.modules) if name == "lib" or name.startswith("lib.")]:
    del sys.modules[module_name]

from lib.llm import LLM
from lib.messages import SystemMessage, ToolMessage, UserMessage
from lib.tooling import tool

In [2]:
chat_model = LLM(provider="claude_code", model="claude-sonnet-4-6", timeout=30)
print(f"Ready: {chat_model.model}")

Ready: claude-sonnet-4-6


## Basic LLM Interaction

Before adding tools, start with two standard interaction patterns: a single text prompt and a structured conversation made from message objects.

In [3]:
response = chat_model.invoke("What is an AI agent?")
print(response.content)

An AI agent is a system where an LLM drives a loop of reasoning and action â€” it perceives inputs, decides what to do, takes actions (often via tools), observes results, and repeats until a goal is met.

The key distinction from a simple LLM call:

- **LLM call**: input â†’ output (one shot)
- **Agent**: input â†’ reason â†’ act â†’ observe â†’ reason â†’ act â†’ ... â†’ output (multi-step loop)

The "tools" are what give agents real-world reach â€” web search, code execution, database queries, API calls, etc. The LLM decides *which* tool to call and *when*, based on what it needs to accomplish the task.


In [4]:
messages = [
    SystemMessage(content="You're an OpenAI API specialist."),
    UserMessage(content="What is function calling?"),
]

response = chat_model.invoke(messages)
print(response.content)

Function calling (also called "tool use") is a feature that allows LLMs to invoke external functions or tools during a conversation, rather than just generating text.

## How it works

1. You define functions with a name, description, and parameter schema (JSON Schema format)
2. You pass those definitions to the model alongside your prompt
3. The model decides when to call a function and returns a structured call with arguments
4. Your code executes the function and returns the result to the model
5. The model uses the result to generate a final response

## Example (OpenAI API)

```python
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get current weather for a location",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {"type": "string", "description": "City name"}
                },
                "required": ["location"]
    

## Build A Tool

A tool is a normal Python function wrapped with `@tool`. The wrapper converts it into the function schema the model receives so it can decide when to call it.

In [5]:
@tool
def get_weather(city: str):
    """Get the current temperature for a city.

    Args:
        city: Name of the city to check weather for.

    Returns:
        A dictionary containing temperature information for the requested city.
    """
    mock_weather = {
        "São Paulo": "28°C",
        "Oslo": "-3°C",
        "New York": "15°C",
        "Tokyo": "22°C",
    }
    return {"temperature": mock_weather.get(city, "Unknown")}

In [6]:
chat_model_with_tools = LLM(provider="claude_code", model="claude-sonnet-4-6", tools=[get_weather], timeout=30)

## Tool Usage Flow

The model does not execute your local Python function by itself. It returns a tool-call request, your code runs the tool, then your code sends a `ToolMessage` back to the model.

The flow is:

1. The user asks a weather question.
2. The assistant responds with a tool call.
3. Your code executes the requested tool.
4. Your code appends a `ToolMessage` with the result.
5. The assistant uses the tool result to produce the final answer.

In [7]:
messages = [
    SystemMessage(
        content=(
            "You are a helpful assistant that can get current temperatures for cities. "
            "Use the weather tool when someone asks about weather or temperature "
            "in a specific location. If the tool does not know, say so."
        )
    ),
    UserMessage(content="How cold is it in Oslo?"),
]

In [8]:
ai_message = chat_model_with_tools.invoke(messages)
messages.append(ai_message)

ai_message

AIMessage(content=None, role='assistant', tool_calls=[namespace(id='call_ebcd89f0', type='function', function=namespace(name='get_weather', arguments='{"city": "Oslo"}'))])

In [9]:
tool_call = ai_message.tool_calls[0]
args = json.loads(tool_call.function.arguments)

tool_result = get_weather(**args)
tool_message = ToolMessage(
    content=tool_result["temperature"],
    tool_call_id=tool_call.id,
    name=tool_call.function.name,
)
messages.append(tool_message)

tool_message

ToolMessage(content='-3°C', role='tool', tool_call_id='call_ebcd89f0', name='get_weather')

In [10]:
final_message = chat_model_with_tools.invoke(messages)
messages.append(final_message)

final_message.content

"It's currently **-3Â°C** in Oslo, so it's quite cold! Make sure to bundle up if you're heading there."

## Inspect The Conversation

The full message list shows each step: system instruction, user request, assistant tool call, tool response, and final assistant answer.

In [11]:
messages

[SystemMessage(content='You are a helpful assistant that can get current temperatures for cities. Use the weather tool when someone asks about weather or temperature in a specific location. If the tool does not know, say so.', role='system'),
 UserMessage(content='How cold is it in Oslo?', role='user'),
 AIMessage(content=None, role='assistant', tool_calls=[namespace(id='call_ebcd89f0', type='function', function=namespace(name='get_weather', arguments='{"city": "Oslo"}'))]),
 ToolMessage(content='-3°C', role='tool', tool_call_id='call_ebcd89f0', name='get_weather'),
 AIMessage(content="It's currently **-3Â°C** in Oslo, so it's quite cold! Make sure to bundle up if you're heading there.", role='assistant', tool_calls=None)]